# PNAD Income Analysis Pipeline

This notebook is the single executable scientific analysis for the project. Numerical procedures are implemented in `pnad_income`; the notebook orchestrates the pipeline, displays the complete analysis, and persists reproducible products under `outputs/`.


## 1. Configuration

The optional legacy 1976–1979 outlier cuts are disabled by default. Setting `APPLY_MANUAL_OUTLIER_CUTS = True` reproduces that historical sensitivity treatment before monetary adjustment and before every downstream statistic and figure.


In [ ]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd

from pnad_income.pipeline import PipelineConfig, pipeline_overview, run_pipeline
from pnad_income.plotting import (
    plot_ccdf, plot_ccdf_grid, plot_extended_inequality_evolution,
    plot_gini_evolution, plot_gini_validation, plot_histogram,
    plot_histogram_grid, plot_lorenz_curve, plot_lorenz_grid,
    plot_measure_comparison, plot_measure_comparison_grid,
    plot_top_income_shares,
)
from pnad_income.outputs import build_diagnostics, export_analysis_outputs, prepare_output_paths
from pnad_income.validation import combine_gini_references, compare_gini_series, gini_validation_statistics

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
DATABASE_PATH = Path(os.environ.get('PNAD_DATABASE_PATH', '../dados_refined')).expanduser()
OUTPUT_ROOT = Path(os.environ.get('PNAD_OUTPUT_PATH', '../outputs')).expanduser()
OUTPUT_PATHS = prepare_output_paths(OUTPUT_ROOT)
REFERENCE_DIR = Path('../config/gini_references')
APPLY_MANUAL_OUTLIER_CUTS = False
CONFIG = PipelineConfig(database_path=DATABASE_PATH, ccdf_base=1.05, start_year=1976, end_year=2025, apply_manual_outlier_cuts=APPLY_MANUAL_OUTLIER_CUTS)
SELECTED_YEAR = 2025
SELECTED_YEARS = [1976, 1990, 2001, 2011, 2020, 2025]
GRID_NROWS = 2
GRID_NCOLS = 3
CONFIG


## 2. Database loading and analytical coverage


In [ ]:
results = run_pipeline(CONFIG)
overview = pipeline_overview(results)
display(overview)
display(results.panel.head())


## 3. Annual descriptive and inequality statistics

The annual table includes Gini, Pietra, the Lorenz intersection $k$, the legacy $Z$ statistic, top-10%, top-1%, and top-0.1% income shares, together with support and missingness diagnostics.


In [ ]:
summary = results.summary
display(summary)


## 4. Plot-selection interface


In [ ]:
plot_histogram(results.panel, year=SELECTED_YEAR, value_col='income', bins=60, yscale='log')
plt.show()
for fig in plot_histogram_grid(results.panel, value_col='income', years=SELECTED_YEARS, bins=60, yscale='log', nrows=GRID_NROWS, ncols=GRID_NCOLS): plt.show()


## 5. Annual Gini coefficient


In [ ]:
plot_gini_evolution(summary, value_col='income')
plt.show()


## 6. Top-income concentration

For top population fraction $q$, the income share is $S_q=1-L(1-q)$. The figure reports $q=0.10$, $0.01$, and $0.001$.


In [ ]:
plot_top_income_shares(summary, value_col='income')
plt.show()


## 7. Pietra, Lorenz intersection k, and legacy Z statistic

The Pietra index is the maximum vertical distance between the equality line and the Lorenz curve. The intersection $k$ satisfies $L(k)=1-k$. The $Z$ statistic reproduces the Lorenz-partition construction in the historical `pnad.py`; its label is retained for reproducibility and its external provenance should be documented separately before substantive interpretation.


In [ ]:
plot_extended_inequality_evolution(summary, value_col='income')
plt.show()


## 8. External Gini validation

Documented IPEA and/or World Bank CSV files placed in `config/gini_references/` are loaded automatically. Hard-coded legacy arrays are not used because their exact indicator provenance is not documented in `pnad.py`.


In [ ]:
reference_files = sorted(REFERENCE_DIR.glob('*.csv'))
gini_references = combine_gini_references(reference_files) if reference_files else pd.DataFrame()
if gini_references.empty:
    print('No documented external Gini reference CSV files are currently installed.')
else:
    gini_comparison = compare_gini_series(summary, gini_references)
    display(gini_comparison)
    display(gini_validation_statistics(gini_comparison))
    plot_gini_validation(summary, gini_references)
    plt.show()


## 9. Annual income histograms — linear frequency scale


In [ ]:
for fig in plot_histogram_grid(results.panel, value_col='income', years=results.years, bins=60, yscale='linear', nrows=6, ncols=4): plt.show()


## 10. Annual income histograms — logarithmic frequency scale


In [ ]:
for fig in plot_histogram_grid(results.panel, value_col='income', years=results.years, bins=60, yscale='log', nrows=6, ncols=4): plt.show()


## 11. Complementary cumulative distribution function

For nonnegative income $X$, $\widehat{\overline F}(x)=N^{-1}\sum_i\mathbf{1}(X_i\ge x)$. Finite zero-income observations remain in the denominator.


In [ ]:
ccdf = results.ccdf_nominal_adjusted
display(ccdf.head(20))


## 12. Individual annual distribution


In [ ]:
plot_ccdf(ccdf, year=SELECTED_YEAR, measure='income', transform='loglog')
plt.show()


## 13. Annual CCDFs — linear axes


In [ ]:
for fig in plot_ccdf_grid(ccdf, measure='income', years=results.years, transform='linear', nrows=6, ncols=4): plt.show()


## 14. Annual CCDFs — log-log axes


In [ ]:
for fig in plot_ccdf_grid(ccdf, measure='income', years=results.years, transform='loglog', nrows=6, ncols=4): plt.show()


## 15. Legacy double-log diagnostic


In [ ]:
for fig in plot_ccdf_grid(ccdf, measure='income', years=results.years, transform='double_log', nrows=6, ncols=4): plt.show()


## 16. Annual Lorenz curves


In [ ]:
for fig in plot_lorenz_grid(results.panel, value_col='income', years=results.years, nrows=6, ncols=4): plt.show()


## 17. Annotated Lorenz curves: G, P, k, Z


In [ ]:
for fig in plot_lorenz_grid(results.panel, value_col='income', years=results.years, nrows=6, ncols=4, annotate=True): plt.show()


## 18. Individual Lorenz curve


In [ ]:
plot_lorenz_curve(results.panel, year=SELECTED_YEAR, value_col='income', annotate=True)
plt.show()


## 19. Nominal versus adjusted distributions


In [ ]:
for fig in plot_measure_comparison_grid(ccdf, measures=('income','income_adj'), years=results.years, transform='loglog', nrows=6, ncols=4): plt.show()
plot_measure_comparison(ccdf, year=SELECTED_YEAR, measures=('income','income_adj'), transform='loglog')
plt.show()


## 20. Effective-income availability


In [ ]:
ccdf_effective = results.ccdf_habitual_effective
if ccdf_effective.empty: print('The current refined database does not contain usable income_effective observations.')
else: display(ccdf_effective.head(20))


## 21. Final data-quality diagnostics


In [ ]:
diagnostics = build_diagnostics(results)
display(diagnostics)


## 22. Persist all scientific outputs


In [ ]:
manifest = export_analysis_outputs(
    results, output_root=OUTPUT_ROOT, selected_year=SELECTED_YEAR,
    selected_years=SELECTED_YEARS, grid_nrows=GRID_NROWS, grid_ncols=GRID_NCOLS,
    complete_nrows=6, complete_ncols=4, histogram_bins=60, dpi=200,
    gini_references=gini_references if not gini_references.empty else None,
)
display(manifest)
print(f'Outputs saved under: {OUTPUT_PATHS.root}')
